In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
from torch_geometric.datasets import QM9
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# ==========================================
# 1. DATA PREPARATION (Exact Paper Math)
# ==========================================
def extract_focal_neighborhoods(data, max_neighbors=7, d_max=1.77):
    num_atoms = data.pos.shape[0]
    dist_matrix = torch.cdist(data.pos, data.pos)
    molecule_neighborhoods = []
    
    for i in range(num_atoms):
        distances = dist_matrix[i]
        sorted_indices = torch.argsort(distances)
        valid_mask = distances[sorted_indices] <= d_max
        valid_indices = sorted_indices[valid_mask]
        
        num_to_take = min(len(valid_indices), max_neighbors + 1)
        neighbor_indices = valid_indices[:num_to_take]
        
        subgraph_z = data.z[neighbor_indices].float()
        subgraph_pos = data.pos[neighbor_indices]
        
        if num_to_take < max_neighbors + 1:
            pad_size = (max_neighbors + 1) - num_to_take
            subgraph_z = torch.cat([subgraph_z, torch.zeros(pad_size)])
            subgraph_pos = torch.cat([subgraph_pos, torch.zeros((pad_size, 3))])
        
        molecule_neighborhoods.append({'atomic_numbers': subgraph_z, 'coordinates': subgraph_pos})
    return molecule_neighborhoods

def encode_diagram_angles(neighborhoods, d_max=1.77):
    processed = []
    for nb in neighborhoods:
        Z_num, pos = nb['atomic_numbers'], nb['coordinates']
        rel_pos = pos - pos[0]
        x, y, z = rel_pos[:, 0], rel_pos[:, 1], rel_pos[:, 2]
        
        d = torch.sqrt(x**2 + y**2 + z**2)
        theta = torch.acos(torch.clamp(y / (d + 1e-7), -1.0, 1.0))
        phi = torch.atan2(x, z)
        
        is_ghost = (Z_num == 0)
        d = d.masked_fill(is_ghost, 0.0)
        theta = theta.masked_fill(is_ghost, 0.0)
        phi = phi.masked_fill(is_ghost, 0.0)
        
        Z_scaled = (Z_num / 10.0) * (2 * np.pi)
        d_scaled = (d / d_max) * (2 * np.pi)
        phi_scaled = ((phi + np.pi) / 2.0).masked_fill(is_ghost, 0.0)
        
        features = torch.stack([Z_scaled, d_scaled, theta, phi_scaled], dim=1)
        processed.append(features.flatten()) 
    return torch.stack(processed)

print("[DEBUG] Loading QM9...")
dataset = QM9(root='./data/QM9')
processed_dataset = []

# Process exactly 10 molecules
for i in range(10):
    data = dataset[i]
    target_val = data.y[0, 4] # HOMO-LUMO Gap
    neighborhoods = extract_focal_neighborhoods(data, d_max=1.77)
    quantum_features = encode_diagram_angles(neighborhoods, d_max=1.77)
    processed_dataset.append((quantum_features, target_val))

# Split: 8 Train, 2 Test
training_data = processed_dataset[:8]
test_data = processed_dataset[8:]
print(f"[DEBUG] Data Ready: {len(training_data)} Train, {len(test_data)} Test.")

C:\Users\Sriram Nangunoori\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DEBUG] Loading QM9...
[DEBUG] Data Ready: 8 Train, 2 Test.


In [2]:
# ==========================================
# 2. QUANTUM CIRCUIT
# ==========================================
num_qubits = 16
num_atoms = 8
num_layers = 8
num_theta = num_qubits * 3 * num_layers # 48 parameters

x_inputs = ParameterVector('x', 32)
theta_weights = ParameterVector('t', num_theta) 
pqc = QuantumCircuit(num_qubits)

# Feature Map
for i in range(num_atoms):
    qA, qB = 2*i, 2*i + 1
    pqc.rx(x_inputs[i*4 + 0], qA)
    pqc.ry(x_inputs[i*4 + 1], qA)
    pqc.rx(x_inputs[i*4 + 2], qB)
    pqc.ry(x_inputs[i*4 + 3], qB)

# Ansatz (1 Layer)
weight_idx = 0
for layer in range(num_layers):
    for q in range(num_qubits):
        pqc.rz(theta_weights[weight_idx], q)
        pqc.ry(theta_weights[weight_idx+1], q)
        pqc.rz(theta_weights[weight_idx+2], q)
        weight_idx += 3
    for q in range(num_qubits):
        pqc.cx(q, (q + 1) % num_qubits)

observables = [SparsePauliOp("I" * (15 - i) + "Z" + "I" * i) for i in range(num_qubits)]
estimator = StatevectorEstimator()

# Explicit parameter list for safe V2 binding
all_params = list(x_inputs) + list(theta_weights)
theta = np.random.uniform(-np.pi, np.pi, num_theta)
quantum_lr = 0.05 

classical_mlp = nn.Sequential(
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
mlp_optimizer = optim.Adam(classical_mlp.parameters(), lr=0.01)
loss_fn = nn.L1Loss()
print(f"[DEBUG] Circuit Built! Total Trainable Quantum Parameters: {num_theta}")

[DEBUG] Circuit Built! Total Trainable Quantum Parameters: 384


In [ ]:
# ==========================================
# 3. MANUAL PARAMETER SHIFT TRAINING LOOP
# ==========================================
print("\n=== STARTING MANUAL 1-EPOCH TRAINING ===")
epochs = 1

for epoch in range(epochs):
    for mol_idx, (X_molecule, y_target) in enumerate(training_data):
        print(f"\n[{time.strftime('%H:%M:%S')}] --- Molecule {mol_idx+1}/8 ---")
        X_np = X_molecule.numpy()
        N_atoms = X_np.shape[0]
        
        # --- 1. QUANTUM FORWARD PASS ---
        Q_out_list = []
        for a in range(N_atoms):
            bound_vals = np.concatenate([X_np[a], theta])
            # Safe Dictionary Binding for Qiskit V2
            param_dict = {p: v for p, v in zip(all_params, bound_vals)}
            job = estimator.run([(pqc, observables, param_dict)])
            Q_out_list.append(job.result()[0].data.evs)
            
        Q_tensor = torch.tensor(np.array(Q_out_list), requires_grad=True, dtype=torch.float32)

        # --- 2. CLASSICAL MLP FORWARD & LOSS ---
        mol_embedding = torch.sum(Q_tensor, dim=0, keepdim=True)
        prediction = classical_mlp(mol_embedding).squeeze()
        y_tensor = y_target.clone().detach().float().squeeze()
        
        loss = loss_fn(prediction, y_tensor)
        print(f"[{time.strftime('%H:%M:%S')}] Forward Pass Done | Loss: {loss.item():.4f}")

        # --- 3. CLASSICAL BACKPROP ---
        mlp_optimizer.zero_grad()
        loss.backward() 
        mlp_optimizer.step() 
        dL_dQ = Q_tensor.grad.numpy() 
        
        # --- 4. MANUAL PARAMETER SHIFT (THE HEAVY LIFTING) ---
        print(f"[{time.strftime('%H:%M:%S')}] Starting Parameter Shift for {num_theta} parameters...")
        manual_theta_grads = np.zeros(num_theta)
        
        for p in range(num_theta):
            theta_plus = theta.copy()
            theta_plus[p] += np.pi / 2.0
            
            theta_minus = theta.copy()
            theta_minus[p] -= np.pi / 2.0
            
            dQ_dtheta_p = np.zeros((N_atoms, 16))
            
            for a in range(N_atoms):
                val_plus = np.concatenate([X_np[a], theta_plus])
                val_minus = np.concatenate([X_np[a], theta_minus])
                
                bind_plus = {param: val for param, val in zip(all_params, val_plus)}
                bind_minus = {param: val for param, val in zip(all_params, val_minus)}
                
                job = estimator.run([
                    (pqc, observables, bind_plus),
                    (pqc, observables, bind_minus)
                ])
                res_plus = job.result()[0].data.evs
                res_minus = job.result()[1].data.evs
                
                dQ_dtheta_p[a] = (res_plus - res_minus) / 2.0
            
            # Chain Rule
            manual_theta_grads[p] = np.sum(dL_dQ * dQ_dtheta_p)
            
            print(f"    -> Shifted {p+1}/{num_theta} parameters...")

        # --- 5. MANUAL WEIGHT UPDATE ---
        theta -= quantum_lr * manual_theta_grads
        print(f"[{time.strftime('%H:%M:%S')}] Molecule {mol_idx+1} Complete! Quantum Weights Updated.")


=== STARTING MANUAL 1-EPOCH TRAINING ===

[02:58:59] --- Molecule 1/8 ---
[02:59:11] Forward Pass Done | Loss: 13.4019
[02:59:11] Starting Parameter Shift for 384 parameters...
    -> Shifted 1/384 parameters...
    -> Shifted 2/384 parameters...
    -> Shifted 3/384 parameters...
    -> Shifted 4/384 parameters...
    -> Shifted 5/384 parameters...
    -> Shifted 6/384 parameters...
    -> Shifted 7/384 parameters...
    -> Shifted 8/384 parameters...
    -> Shifted 9/384 parameters...
    -> Shifted 10/384 parameters...
    -> Shifted 11/384 parameters...
    -> Shifted 12/384 parameters...
    -> Shifted 13/384 parameters...
    -> Shifted 14/384 parameters...
    -> Shifted 15/384 parameters...
    -> Shifted 16/384 parameters...
    -> Shifted 17/384 parameters...
    -> Shifted 18/384 parameters...
    -> Shifted 19/384 parameters...
    -> Shifted 20/384 parameters...
    -> Shifted 21/384 parameters...
    -> Shifted 22/384 parameters...
    -> Shifted 23/384 parameters...
   

In [5]:
# ==========================================
# 4. MANUAL TESTING LOOP (2 Molecules)
# ==========================================
print("\n=== STARTING EVALUATION ON TEST SET ===")
classical_mlp.eval()

test_loss = 0.0
predictions = []
actuals = []

with torch.no_grad(): # Disable MLP gradients
    for mol_idx, (X_molecule, y_target) in enumerate(test_data):
        X_np = X_molecule.numpy()
        N_atoms = X_np.shape[0]
        
        # Manual Quantum Forward Pass
        Q_out_list = []
        for a in range(N_atoms):
            bound_vals = np.concatenate([X_np[a], theta])
            param_dict = {p: v for p, v in zip(all_params, bound_vals)}
            job = estimator.run([(pqc, observables, param_dict)])
            Q_out_list.append(job.result()[0].data.evs)
            
        Q_tensor = torch.tensor(np.array(Q_out_list), dtype=torch.float32)
        
        # Classical Forward Pass
        mol_embedding = torch.sum(Q_tensor, dim=0, keepdim=True)
        prediction = classical_mlp(mol_embedding).squeeze()
        y_tensor = y_target.clone().detach().float().squeeze()
        
        loss = loss_fn(prediction, y_tensor)
        test_loss += loss.item()
        
        predictions.append(prediction.item())
        actuals.append(y_tensor.item())
        
        print(f"Test Molecule {mol_idx+1}/2 | Actual: {actuals[-1]:.4f} | Predicted: {predictions[-1]:.4f} | Error: {loss.item():.4f}")

print(f"\n==> Final Test MAE: {test_loss / len(test_data):.4f} Hartrees")


=== STARTING EVALUATION ON TEST SET ===
Test Molecule 1/2 | Actual: 8.7675 | Predicted: 0.8639 | Error: 7.9036
Test Molecule 2/2 | Actual: 9.9049 | Predicted: 0.7407 | Error: 9.1643

==> Final Test MAE: 8.5339 Hartrees
